In [ ]:
import os
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
import torch
import nltk
from nltk.corpus import stopwords
from pymorphy2 import MorphAnalyzer

nltk.download('stopwords')

# Конфигурация
DATA_DIR = "data"
FILES = [
    "Biblioteka_prikluchenij.txt",
    "detective_for_kidds.txt",
    "detective_masters.txt",
    "russian_love_story.txt"
]

# Инициализация инструментов для русского языка
morph = MorphAnalyzer()
russian_stopwords = stopwords.words('russian')

# Функция предобработки текста
def preprocess_text(text):
    # Очистка текста
    text = re.sub(r'[^\w\s]', '', text.lower())
    # Токенизация
    words = text.split()
    # Лемматизация и удаление стоп-слов
    return [morph.parse(word)[0].normal_form for word in words 
            if word not in russian_stopwords and len(word) > 2]

# Загрузка и обработка данных
sentences = []
for file in FILES:
    with open(os.path.join(DATA_DIR, file), 'r', encoding='utf-8') as f:
        text = f.read()
        sentences += [preprocess_text(text)]

print(f"Загружено {len(sentences)} документов")

# Метод 1: Word2Vec
print("\n" + "="*40 + "\nОбучение Word2Vec модели\n" + "="*40)
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=3,
    workers=4
)

# Пример использования
test_word = "детектив"
print(f"\nВектор для слова '{test_word}':")
print(w2v_model.wv[test_word].shape)
print(w2v_model.wv[test_word][:10])

# Метод 2: Предобученная модель из Hugging Face
print("\n" + "="*40 + "\nИспользование ruBERT\n" + "="*40)
model_name = "sberbank-ai/ruBert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_bert_embedding(word):
    inputs = tokenizer(word, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return torch.mean(outputs.last_hidden_state, dim=1).squeeze().numpy()

# Пример использования
bert_vector = get_bert_embedding(test_word)
print(f"\nВектор BERT для слова '{test_word}':")
print(bert_vector.shape)
print(bert_vector[:10])

# Метод 3: Частотное представление (TF-IDF)
print("\n" + "="*40 + "\nСоздание TF-IDF векторов\n" + "="*40)
# Преобразуем токены обратно в строки для TF-IDF
processed_texts = [' '.join(doc) for doc in sentences]

tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(processed_texts)

# Пример использования
word_index = tfidf.vocabulary_[test_word]
print(f"\nTF-IDF вектор для слова '{test_word}':")
print(f"Индекс в матрице: {word_index}")
print(f"TF-IDF вес: {tfidf.idf_[word_index]:.4f}")

# Сравнение результатов
print("\n" + "="*40 + "\nСравнение представлений\n" + "="*40)
print(f"Слово: {test_word}")
print(f"Word2Vec размер: {w2v_model.wv.vector_size}")
print(f"BERT размер: {bert_vector.shape[0]}")
print(f"TF-IDF общий размер: {tfidf_matrix.shape[1]}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nmens\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
